# C04 — Inferência Local vs. Remota

**Disciplina:** Sistemas Cognitivos com Large Language Models (INFNET, 26E2_3)
**Autor:** Anderson Felipe Paixão Corrêa
**Projeto:** O Direito como Dado — RAG hierarquia- e vigência-aware sobre o microssistema
penal federal brasileiro

Esta notebook cobre o **ponto 4 da rubrica** (inferência local ou remota). Ela:

1. **Mede** a inferência **local** já usada em produção neste projeto (Ollama +
   `llama3.1:8b`, o mesmo backend de `c05_rag_pipeline.ipynb` e `c06_antinomias.ipynb`) —
   latência real, em segundos, de chamadas de geração de verdade.
2. **Discute** (sem executar) um baseline em **nuvem** — a API OpenAI/Anthropic teria o
   mesmo contrato `LLMClient`, mas nenhuma chave é lida, exigida ou commitada aqui.
3. **Propõe** a arquitetura de produção — **nuvem privada** com pesos abertos.
4. Fecha com uma **tabela comparativa** (privacidade / qualidade / custo / latência /
   controle) usando a latência local medida na Seção 1 e estimativas raciocinadas e
   claramente rotuladas para os demais cenários.


In [1]:
import sys
import time
from pathlib import Path

_repo_root = Path.cwd()
if not (_repo_root / "direito_dados").exists():
    _repo_root = Path(__file__).resolve().parent if "__file__" in globals() else _repo_root
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from direito_dados.generation.llm import LLMClient, OllamaClient, ollama_available, ollama_has_model

MODEL = "llama3.1:8b"
assert ollama_available(), "Ollama precisa estar rodando em localhost:11434"
assert ollama_has_model(MODEL), f"Modelo {MODEL} precisa estar baixado (`ollama pull {MODEL}`)"
print(f"Ollama ativo, modelo {MODEL} disponível.")


Ollama ativo, modelo llama3.1:8b disponível.


## 1. Inferência local (Ollama + `llama3.1:8b`)

**Por que local, primeiro:** consultas sobre Direito Penal podem envolver detalhes de casos
reais (fatos, nomes, contexto de uma situação concreta que alguém está tentando entender).
Rodando inteiramente na máquina — modelo, prompt, geração — **nenhum dado trafega pela
rede nem passa por um terceiro**. Sob a LGPD (Lei 13.709/2018), isso elimina de saída toda a
superfície de risco associada a compartilhar dados pessoais/sensíveis com um processador
externo: não há política de retenção de terceiro para auditar, não há log fora do nosso
controle, não há necessidade de base legal para transferência de dados a um provedor.

Medimos abaixo três chamadas reais de geração, cronometradas com `time.time()`: uma pergunta
curta em prosa livre, uma pergunta comparativa (mais tokens de saída) e uma chamada em modo
JSON estruturado — o mesmo modo (`json_format=True`) usado pelo pipeline RAG em produção
(`answer_question`, `c05_rag_pipeline.ipynb`).


In [2]:
latencies: list[tuple[str, float]] = []


def timed_generate(client, label, prompt, **kwargs):
    t0 = time.time()
    resp = client.generate(prompt, **kwargs)
    dt = time.time() - t0
    latencies.append((label, dt))
    print(f"[{label}] {dt:.2f}s")
    print(resp)
    print()
    return resp


llm_prose = OllamaClient(model=MODEL, json_format=False)
llm_json = OllamaClient(model=MODEL)  # json_format=True (padrão, o mesmo modo do pipeline RAG)


### Chamada 1 — pergunta curta, prosa livre

In [3]:
_ = timed_generate(
    llm_prose,
    "prosa curta",
    "Em uma frase, o que caracteriza o crime de furto no Código Penal brasileiro (art. 155)?",
)


[prosa curta] 2.21s
O crime de furto é caracterizado quando a subtração da coisa móvel alheia ocorre mediante grave dano à propriedade ou a pessoa da vítima e, ainda, se o agente age com violência contra pessoa ou o emprego de arma de fogo.



### Chamada 2 — pergunta comparativa, prosa livre, mais tokens de saída

In [4]:
_ = timed_generate(
    llm_prose,
    "prosa longa",
    "Explique em até três frases a diferença entre furto (art. 155) e roubo (art. 157) "
    "do Código Penal brasileiro.",
)


[prosa longa] 2.56s
O crime de furto ocorre quando alguém retira um bem móvel, sem violência ou ameaça. Já o roubo se caracteriza pela retirada de um bem móvel, mediante violência ou ameaça. Nesse último caso, existe uma ação mais agressiva do autor em relação ao lesado.



### Chamada 3 — saída estruturada em JSON (o modo usado pelo pipeline RAG em produção)


In [5]:
schema = {
    "type": "object",
    "properties": {"resposta": {"type": "string"}, "artigo": {"type": "string"}},
    "required": ["resposta", "artigo"],
}
_ = timed_generate(
    llm_json,
    "json estruturado",
    "Qual artigo do Código Penal tipifica o crime de peculato e qual a pena? "
    "Responda em JSON com as chaves 'resposta' e 'artigo'.",
    format=schema,
)


[json estruturado] 3.46s
{
"resposta": "Artigo 312 do Código Penal",
"artigo": "O crime de peculato é tipificado pelo artigo 312 do Código Penal, que prevê a seguinte pena: pena de reclusão, desde que o valor do objeto for superior a R$ 8.457,38 (oito mil quatrocentos e cinquenta e sete reais e trinta e oito centavos)."
}



### Latências medidas

In [6]:
from IPython.display import Markdown, display

avg_latency = sum(dt for _, dt in latencies) / len(latencies)
rows = "\n".join(f"| {label} | {dt:.2f}s |" for label, dt in latencies)
display(Markdown(f"| Chamada | Latência |\n|---|---|\n{rows}\n| **Média** | **{avg_latency:.2f}s** |"))
print(f"Latência média local ({MODEL}, {len(latencies)} chamadas): {avg_latency:.2f}s")


| Chamada | Latência |
|---|---|
| prosa curta | 2.21s |
| prosa longa | 2.56s |
| json estruturado | 3.46s |
| **Média** | **2.74s** |

Latência média local (llama3.1:8b, 3 chamadas): 2.74s


**Leitura:** as três chamadas acima rodaram inteiramente nesta máquina — nenhuma consulta
sobre os artigos do Código Penal trafegou pela rede. A latência medida (poucos segundos por
chamada, nesta máquina com o modelo já carregado em memória) é referenciada na tabela
comparativa da Seção 4, junto com estimativas para os cenários em nuvem.

Vale notar, como aparte: a Chamada 3 (peculato) foi feita **sem** contexto recuperado do
corpus — só a pergunta, sem grounding. Se o artigo/pena que o modelo devolveu não bater com
o real (CP art. 312), isso não é falha da medição de latência, é exatamente o comportamento
que o pipeline RAG do projeto (`c05_rag_pipeline.ipynb`) foi desenhado para não deixar passar
sem verificação: toda citação que o modelo gera é conferida contra o corpus antes de ser
aceita (`answer_question`), e aqui — chamando o modelo "pelado" — não há essa checagem.


## 2. Baseline em nuvem (arquitetura, não executado)

`LLMClient` (`direito_dados.generation.llm`) é um `Protocol` — qualquer classe com um método
`generate(prompt, system=None, format=None) -> str` é compatível, sem precisar herdar de
nada. Isso significa que um cliente de nuvem (OpenAI, Anthropic, ou qualquer API compatível)
poderia substituir o `OllamaClient` em `answer_question` ou `detect_conflicts` **sem
nenhuma mudança** nesses módulos.

Abaixo, um esqueleto de `CloudClient` que satisfaz esse contrato. Ele é **arquitetado, não
executado**: `.generate()` levanta `NotImplementedError` de propósito, e nenhuma chave de
API é lida, exigida ou commitada neste repositório.


In [7]:
import os


class CloudClient:
    """Esqueleto de um cliente de LLM em nuvem (ex.: OpenAI/Anthropic) que satisfaz o
    mesmo protocolo `LLMClient` usado por `OllamaClient`.

    Não é chamado nesta notebook — existe só para mostrar que a arquitetura já suporta um
    backend em nuvem por composição (Protocol), não por herança. Trocar `OllamaClient` por
    este cliente em `answer_question`/`detect_conflicts` não exigiria alterar essas funções.
    """

    def __init__(self, model: str, api_key_env: str = "OPENAI_API_KEY"):
        self.model = model
        self.api_key_env = api_key_env

    def generate(self, prompt: str, system: str | None = None, format=None) -> str:
        raise NotImplementedError(
            "Arquitetado, não executado nesta entrega — chamaria a API do provedor "
            f"(lendo a chave de {self.api_key_env}) com o mesmo contrato de "
            "LLMClient.generate()."
        )


cloud = CloudClient(model="gpt-4o-mini")
print("CloudClient satisfaz o protocolo LLMClient:", isinstance(cloud, LLMClient))
print(f"Variável de ambiente {cloud.api_key_env!r} está configurada?",
      os.environ.get(cloud.api_key_env) is not None)


CloudClient satisfaz o protocolo LLMClient: True
Variável de ambiente 'OPENAI_API_KEY' está configurada? False


**Tradeoffs de um baseline em nuvem** (raciocinados, não medidos aqui):

- **Qualidade:** modelos de fronteira proprietários costumam ter mais capacidade de
  raciocínio e conhecimento mais amplo/atualizado que um modelo aberto de 8B rodando
  localmente — plausivelmente responderiam melhor a perguntas jurídicas fora do que foi
  recuperado pelo RAG (o que, aliás, é o comportamento que o projeto tenta evitar — ver
  Seção 1 de `c01_modelos_llm.ipynb`).
- **Custo:** cobrança por token, sem custo de hardware upfront — mas escala linearmente com
  o volume de consultas; em produção com muitos usuários pode superar o custo de possuir a
  infraestrutura.
- **Latência:** round-trip de rede + fila do provedor somam-se ao tempo de inferência;
  tende a ser maior que a inferência local para prompts curtos, ainda que a infraestrutura
  do provedor seja rápida.
- **Privacidade/controle:** a consulta sai da máquina e passa pela infraestrutura de um
  terceiro. Mesmo com política de "não usar para treinamento", isso normalmente exige um
  acordo comercial (enterprise/zero-retention) para ter garantia contratual — algo que este
  projeto, na forma atual, não tem. Sob a LGPD, isso implica base legal para o
  compartilhamento e depender da política de retenção do provedor.


## 3. Arquitetura de nuvem privada (produção)

A resposta de produção não é "local na máquina do desenvolvedor" nem "API pública de
terceiro" — é um meio-termo que preserva o essencial da inferência local (nenhum dado sai do
perímetro de confiança) com a elasticidade de operar em nuvem:

- **Pesos abertos, self-hosted:** servir `llama3.1` (ou um modelo maior, se o caso de uso
  justificar) via **vLLM** ou **Ollama** rodando em instâncias próprias — o mesmo modelo,
  sem mudar o contrato `LLMClient.generate()`.
- **Rede isolada:** o serviço de inferência roda dentro de uma **VPC privada**, sem endpoint
  público; só o `direito_dados` (ou o serviço que o expõe) tem acesso de rede a ele.
- **Residência de dados no Brasil:** hospedar a região/datacenter no Brasil atende
  diretamente o espírito da LGPD para dados sensíveis de natureza jurídica — sem
  transferência internacional de dados a justificar.
- **Sem logging de terceiro:** ao contrário de uma API pública, não há processador de dados
  externo guardando prompts/respostas — o log (se houver, para auditoria/depuração) fica sob
  controle do próprio projeto, com retenção e acesso definidos por nós.
- **Escala sob demanda:** autoscaling de GPU absorve picos de uso sem exigir hardware ocioso
  o tempo todo (o principal ganho sobre "só rodar local na máquina de um usuário").

Essa é a arquitetura que este projeto adotaria para produção real: o `LLMClient` já abstrai
o backend, então migrar de "Ollama na máquina local" para "vLLM numa VPC" é uma troca de
`host`/deploy, não uma reescrita do pipeline RAG ou do detector de antinomias.


## 4. Comparação: local vs. nuvem vs. nuvem privada

In [8]:
comparativo = f"""
| Critério | Local (Ollama, `llama3.1:8b`) | Nuvem (API OpenAI/Anthropic) | Nuvem privada (self-hosted, VPC) |
|---|---|---|---|
| Privacidade | **Máxima** — nenhum dado sai da máquina | Dados trafegam para o provedor; sujeitos à política de retenção/treinamento dele, salvo acordo enterprise | Alta — dados ficam na VPC própria, sem terceiro processando |
| Qualidade | Boa para respostas fundamentadas em RAG; modelo 8B tem limites de raciocínio fora do que foi recuperado | Geralmente superior (modelos maiores, mais atualizados) — *estimativa raciocinada* | Configurável — pode hospedar modelos maiores que 8B se houver GPU suficiente — *estimativa raciocinada* |
| Custo | Sem custo por chamada (hardware já disponível/amortizado) | Por token, escala com volume — pode superar o custo local em produção com muitos usuários — *estimativa raciocinada* | Custo de infraestrutura (GPU sob demanda/autoscaling) — maior custo fixo, sem custo marginal por token — *estimativa raciocinada* |
| Latência | **Medida nesta notebook: {avg_latency:.2f}s em média** ({len(latencies)} chamadas, Seção 1) | Rede + fila do provedor somam-se à inferência; tipicamente maior que local para prompts curtos — *estimativa raciocinada* | Depende da rede interna da VPC; pode se aproximar da latência local se bem dimensionada — *estimativa raciocinada* |
| Controle | Total (versão do modelo, prompt, dados, disponibilidade) | Baixo — dependente da disponibilidade/preço/deprecação do provedor | Alto — infraestrutura própria, mesma interface `LLMClient` do resto do projeto |
"""
from IPython.display import Markdown, display

display(Markdown(comparativo))



| Critério | Local (Ollama, `llama3.1:8b`) | Nuvem (API OpenAI/Anthropic) | Nuvem privada (self-hosted, VPC) |
|---|---|---|---|
| Privacidade | **Máxima** — nenhum dado sai da máquina | Dados trafegam para o provedor; sujeitos à política de retenção/treinamento dele, salvo acordo enterprise | Alta — dados ficam na VPC própria, sem terceiro processando |
| Qualidade | Boa para respostas fundamentadas em RAG; modelo 8B tem limites de raciocínio fora do que foi recuperado | Geralmente superior (modelos maiores, mais atualizados) — *estimativa raciocinada* | Configurável — pode hospedar modelos maiores que 8B se houver GPU suficiente — *estimativa raciocinada* |
| Custo | Sem custo por chamada (hardware já disponível/amortizado) | Por token, escala com volume — pode superar o custo local em produção com muitos usuários — *estimativa raciocinada* | Custo de infraestrutura (GPU sob demanda/autoscaling) — maior custo fixo, sem custo marginal por token — *estimativa raciocinada* |
| Latência | **Medida nesta notebook: 2.74s em média** (3 chamadas, Seção 1) | Rede + fila do provedor somam-se à inferência; tipicamente maior que local para prompts curtos — *estimativa raciocinada* | Depende da rede interna da VPC; pode se aproximar da latência local se bem dimensionada — *estimativa raciocinada* |
| Controle | Total (versão do modelo, prompt, dados, disponibilidade) | Baixo — dependente da disponibilidade/preço/deprecação do provedor | Alto — infraestrutura própria, mesma interface `LLMClient` do resto do projeto |


## Conclusão

A inferência local (Ollama + `llama3.1:8b`) é a escolha certa para este projeto **hoje**:
latência medida de poucos segundos por chamada (Seção 1), custo marginal zero, e a
propriedade mais importante para consultas de Direito Penal — nenhum dado sai da máquina,
o que resolve de saída a maior parte da superfície de risco sob a LGPD. Um baseline em
nuvem (Seção 2) trocaria essa privacidade por potencial ganho de qualidade/velocidade de
infraestrutura, a um custo recorrente e com dependência de terceiro — arquitetado aqui via
o mesmo protocolo `LLMClient`, mas não executado (nenhuma chave commitada). A resposta de
produção (Seção 3) não escolhe um extremo: **nuvem privada, pesos abertos, residência de
dados no Brasil** preserva o essencial da privacidade local com a elasticidade de nuvem — e,
por construção (`LLMClient` como `Protocol`), a migração entre os três cenários não exige
reescrever `answer_question` nem `detect_conflicts`, só trocar qual cliente é instanciado.
